# Notebook 29 — Replay Dataset Analysis and Opponent Diversity

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 28 generated 100 self-play games and 350 structured replay steps.

Notebook 29 analyzes that replay dataset, measures repetition and class imbalance, compares outcomes by starting side, and prepares more diverse opponent configurations for future self-play training.

## Objectives

1. Load Notebook 28 artifacts.
2. Validate replay dataset integrity.
3. Analyze winner imbalance.
4. Measure move repetition.
5. Compare game length and score distributions.
6. Examine starting-side effects.
7. Identify duplicated training patterns.
8. Define diverse opponent profiles.
9. Generate an opponent-diversity experiment plan.
10. Export analysis reports for Notebook 30.

# Cell 2 — Imports

In [1]:
from __future__ import annotations

import json
import sys

from pathlib import Path
from typing import Any

import pandas as pd

print("Python:", sys.version)
print("Notebook 29 initialized.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Notebook 29 initialized.


# Cell 3 — Locate the Project and Reports

In [2]:
from pathlib import Path


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = {
        "notebooks",
        "scripts",
        "src",
        "reports",
    }

    for candidate in [
        current,
        *current.parents,
    ]:
        if all(
            (candidate / marker).exists()
            for marker in markers
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found."
    )


PROJECT_ROOT = find_project_root()

NOTEBOOK28_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook28"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook29"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Notebook 28 reports:", NOTEBOOK28_REPORT_DIR)
print("Notebook 29 reports:", REPORT_DIR)

assert NOTEBOOK28_REPORT_DIR.exists()

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 28 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook28
Notebook 29 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29


# Cell 4 — Load Notebook 28 Artifacts

In [3]:
DATASET_FILE = NOTEBOOK28_REPORT_DIR / "self_play_dataset.csv"
STATS_FILE = NOTEBOOK28_REPORT_DIR / "self_play_statistics.csv"
SUMMARY_FILE = NOTEBOOK28_REPORT_DIR / "self_play_summary.json"

assert DATASET_FILE.exists(), DATASET_FILE
assert STATS_FILE.exists(), STATS_FILE
assert SUMMARY_FILE.exists(), SUMMARY_FILE

dataset = pd.read_csv(DATASET_FILE)
statistics = pd.read_csv(STATS_FILE)

with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
    summary = json.load(f)

print("=" * 60)
print("Notebook 28 Artifacts Loaded")
print("=" * 60)

print(f"Dataset rows : {len(dataset):,}")
print(f"Statistics rows : {len(statistics):,}")
print()

print("Summary")
for key, value in summary.items():
    print(f"{key:20} {value}")
    

Notebook 28 Artifacts Loaded
Dataset rows : 350
Statistics rows : 100

Summary
notebook             28
project              PTCG AI Battle Challenge
team                 Team Jesus
session_id           f49b7dce-dd65-4367-abb0-5210ad4e8783
random_seed          42
requested_games      100
completed_games      100
total_replay_steps   350
average_turns        3.5
average_score        155.0
winner_counts        {'TrainingAgent': 100}
parse_failures       []
elapsed_seconds      0.33350229999632575


# Cell 5 — Dataset Overview

In [4]:
print("=" * 60)
print("Dataset Overview")
print("=" * 60)

display(dataset.head())

print()

print(dataset.info())

print()

print(dataset.describe(include="all"))

Dataset Overview


,game_id,winner,turn,player,chosen_move,evaluation,search_depth,num_legal_moves,state
0,a867f751-bbdf-4999-8c1c-392f2a803324,TrainingAgent,1,Player,Evolution Burst,633.0,6,0,"{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes..."
1,a867f751-bbdf-4999-8c1c-392f2a803324,TrainingAgent,2,Opponent,Thunder Jolt,-633.0,6,0,"{'pokemon': 'Electrike', 'damage': 30.0, 'node..."
2,a867f751-bbdf-4999-8c1c-392f2a803324,TrainingAgent,3,Player,Evolution Burst,633.0,6,0,"{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes..."
3,148580ca-739b-4be8-a824-5d7934c420c6,TrainingAgent,1,Opponent,Thunder Jolt,-618.0,6,0,"{'pokemon': 'Electrike', 'damage': 30.0, 'node..."
4,148580ca-739b-4be8-a824-5d7934c420c6,TrainingAgent,2,Player,Evolution Burst,618.0,6,0,"{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes..."



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   game_id          350 non-null    object 
 1   winner           350 non-null    object 
 2   turn             350 non-null    int64  
 3   player           350 non-null    object 
 4   chosen_move      350 non-null    object 
 5   evaluation       350 non-null    float64
 6   search_depth     350 non-null    int64  
 7   num_legal_moves  350 non-null    int64  
 8   state            350 non-null    object 
dtypes: float64(1), int64(3), object(5)
memory usage: 24.7+ KB
None

                                     game_id         winner        turn  \
count                                    350            350  350.000000   
unique                                   100              1         NaN   
top     148580ca-739b-4be8-a824-5d7934c420c6  TrainingAgent         NaN   
freq                

# Cell 6 — Data Quality Report

In [5]:
print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(f"Games                : {dataset.game_id.nunique():>6}")
print(f"Replay steps         : {len(dataset):>6}")
print(f"Unique winners       : {dataset.winner.nunique():>6}")
print(f"Unique players       : {dataset.player.nunique():>6}")
print(f"Unique moves         : {dataset.chosen_move.nunique():>6}")
print(f"Unique states        : {dataset.state.nunique():>6}")
print(f"Missing values       : {dataset.isna().sum().sum():>6}")

print()

print("Winner Distribution")
print(dataset["winner"].value_counts())

print()

print("Player Distribution")
print(dataset["player"].value_counts())

print()

print("Move Distribution")
print(dataset["chosen_move"].value_counts())

DATA QUALITY REPORT
Games                :    100
Replay steps         :    350
Unique winners       :      1
Unique players       :      2
Unique moves         :      2
Unique states        :      7
Missing values       :      0

Winner Distribution
winner
TrainingAgent    350
Name: count, dtype: int64

Player Distribution
player
Player      200
Opponent    150
Name: count, dtype: int64

Move Distribution
chosen_move
Evolution Burst    200
Thunder Jolt       150
Name: count, dtype: int64


# Cell 7 — Replay Diversity Metrics

In [6]:
print("=" * 70)
print("Replay Diversity Metrics")
print("=" * 70)

games = dataset.game_id.nunique()

steps_per_game = len(dataset) / games

unique_states = dataset.state.nunique()

unique_moves = dataset.chosen_move.nunique()

print(f"Average steps/game : {steps_per_game:.2f}")
print(f"Unique states      : {unique_states}")
print(f"Unique moves       : {unique_moves}")

print()

state_diversity = unique_states / len(dataset)

move_diversity = unique_moves / len(dataset)

print(f"State diversity : {state_diversity:.3f}")
print(f"Move diversity  : {move_diversity:.3f}")

print()

if state_diversity < 0.10:
    print("⚠ Low state diversity")

if move_diversity < 0.05:
    print("⚠ Low move diversity")

Replay Diversity Metrics
Average steps/game : 3.50
Unique states      : 7
Unique moves       : 2

State diversity : 0.020
Move diversity  : 0.006

⚠ Low state diversity
⚠ Low move diversity


# Cell 8 — Analyze Game Length

In [7]:
print("=" * 70)
print("Game Length Analysis")
print("=" * 70)

turns_per_game = (
    dataset.groupby("game_id")["turn"]
    .max()
    .sort_values()
)

display(turns_per_game.describe())

print()

print("Turn Frequency")
print(turns_per_game.value_counts().sort_index())

print()

print(f"Shortest game : {turns_per_game.min()} turns")
print(f"Longest game  : {turns_per_game.max()} turns")
print(f"Average length: {turns_per_game.mean():.2f} turns")

Game Length Analysis


count    100.000000
mean       3.500000
std        0.502519
min        3.000000
25%        3.000000
50%        3.500000
75%        4.000000
max        4.000000
Name: turn, dtype: float64


Turn Frequency
turn
3    50
4    50
Name: count, dtype: int64

Shortest game : 3 turns
Longest game  : 4 turns
Average length: 3.50 turns


# Cell 9 — Move Frequency Analysis

In [8]:
print("=" * 70)
print("Move Frequency Analysis")
print("=" * 70)

move_counts = (
    dataset["chosen_move"]
    .value_counts()
    .rename_axis("Move")
    .reset_index(name="Count")
)

display(move_counts)

move_counts["Percent"] = (
    move_counts["Count"]
    / len(dataset)
    * 100
)

display(move_counts)

Move Frequency Analysis


,Move,Count
0,Evolution Burst,200
1,Thunder Jolt,150


,Move,Count,Percent
0,Evolution Burst,200,57.142857
1,Thunder Jolt,150,42.857143


# Cell 10 — Player Analysis

In [9]:
print("=" * 70)
print("Player Analysis")
print("=" * 70)

player_summary = (
    dataset
    .groupby("player")
    .agg(
        Steps=("turn", "count"),
        AvgEvaluation=("evaluation", "mean"),
        AvgTurn=("turn", "mean"),
    )
)

display(player_summary)


Player Analysis


,Steps,AvgEvaluation,AvgTurn
player,,,
Opponent,150,-623.0,2.0
Player,200,625.5,2.5


# Cell 11 — Duplicate State Analysis

In [10]:
print("=" * 70)
print("Duplicate State Analysis")
print("=" * 70)

state_counts = dataset["state"].value_counts()

display(state_counts)

duplicate_states = (state_counts > 1).sum()

print()

print(f"Unique states      : {dataset.state.nunique()}")
print(f"Duplicate patterns : {duplicate_states}")
print(f"Most common state appears {state_counts.max()} times")

Duplicate State Analysis


state
{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes_searched': 17, 'player_hp_after': 200.0, 'opponent_hp_after': 10.0, 'next_side': 'Opponent'}    50
{'pokemon': 'Electrike', 'damage': 30.0, 'nodes_searched': 10, 'player_hp_after': 170.0, 'opponent_hp_after': 10.0, 'next_side': 'Player'}     50
{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes_searched': 10, 'player_hp_after': 170.0, 'opponent_hp_after': 0.0, 'next_side': 'Opponent'}     50
{'pokemon': 'Electrike', 'damage': 30.0, 'nodes_searched': 16, 'player_hp_after': 170.0, 'opponent_hp_after': 70.0, 'next_side': 'Player'}     50
{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes_searched': 17, 'player_hp_after': 170.0, 'opponent_hp_after': 10.0, 'next_side': 'Opponent'}    50
{'pokemon': 'Electrike', 'damage': 30.0, 'nodes_searched': 10, 'player_hp_after': 140.0, 'opponent_hp_after': 10.0, 'next_side': 'Player'}     50
{'pokemon': 'Eevee ex', 'damage': 60.0, 'nodes_searched': 10, 'player_hp_after': 140.0, 'opponent_hp_after': 0.0, 'nex


Unique states      : 7
Duplicate patterns : 7
Most common state appears 50 times


# Cell 12 — Match-Count Experiment Plan

In [11]:
experiment_plan = pd.DataFrame(
    [
        {
            "experiment": "smoke_test",
            "num_matches": 100,
            "purpose": "Validate pipeline and exports",
        },
        {
            "experiment": "analysis_run",
            "num_matches": 1_000,
            "purpose": "Measure stability and repetition",
        },
        {
            "experiment": "full_training_run",
            "num_matches": 2_000,
            "purpose": "Generate training data after opponent diversity",
        },
    ]
)

display(experiment_plan)

,experiment,num_matches,purpose
0,smoke_test,100,Validate pipeline and exports
1,analysis_run,1000,Measure stability and repetition
2,full_training_run,2000,Generate training data after opponent diversity


# Cell 13 — Opponent Allocation Plan

In [12]:
opponent_allocation = pd.DataFrame(
    [
        {
            "opponent_profile": "random",
            "num_matches": 400,
            "purpose": "Increase action and state variety",
        },
        {
            "opponent_profile": "aggressive",
            "num_matches": 400,
            "purpose": "Test damage-first strategies",
        },
        {
            "opponent_profile": "defensive",
            "num_matches": 400,
            "purpose": "Test survival and resource conservation",
        },
        {
            "opponent_profile": "greedy",
            "num_matches": 400,
            "purpose": "Test immediate-value decisions",
        },
        {
            "opponent_profile": "search_based",
            "num_matches": 400,
            "purpose": "Test stronger calculated play",
        },
    ]
)

assert opponent_allocation["num_matches"].sum() == 2_000

display(opponent_allocation)

print(
    "Total planned matches:",
    opponent_allocation["num_matches"].sum(),
)

,opponent_profile,num_matches,purpose
0,random,400,Increase action and state variety
1,aggressive,400,Test damage-first strategies
2,defensive,400,Test survival and resource conservation
3,greedy,400,Test immediate-value decisions
4,search_based,400,Test stronger calculated play


Total planned matches: 2000


# Cell 14 — Create the Notebook 29 Analysis Summary

In [13]:
analysis_summary = {
    "source_notebook": 28,
    "total_games": int(dataset["game_id"].nunique()),
    "total_steps": int(len(dataset)),
    "unique_winners": int(dataset["winner"].nunique()),
    "unique_players": int(dataset["player"].nunique()),
    "unique_moves": int(dataset["chosen_move"].nunique()),
    "unique_states": int(dataset["state"].nunique()),
    "state_diversity": float(state_diversity),
    "move_diversity": float(move_diversity),
    "average_steps_per_game": float(steps_per_game),
    "duplicate_state_patterns": int(duplicate_states),
    "most_common_state_frequency": int(state_counts.max()),
    "recommended_analysis_matches": 1_000,
    "recommended_training_matches": 2_000,
    "recommendation": (
        "Introduce opponent diversity before generating "
        "the 2,000-game training dataset."
    ),
}

for key, value in analysis_summary.items():
    print(f"{key:32}: {value}")

source_notebook                 : 28
total_games                     : 100
total_steps                     : 350
unique_winners                  : 1
unique_players                  : 2
unique_moves                    : 2
unique_states                   : 7
state_diversity                 : 0.02
move_diversity                  : 0.005714285714285714
average_steps_per_game          : 3.5
duplicate_state_patterns        : 7
most_common_state_frequency     : 50
recommended_analysis_matches    : 1000
recommended_training_matches    : 2000
recommendation                  : Introduce opponent diversity before generating the 2,000-game training dataset.


# Cell 15 — Export Notebook 29 Reports

In [14]:
analysis_summary_file = (
    REPORT_DIR
    / "replay_analysis_summary.json"
)

experiment_plan_file = (
    REPORT_DIR
    / "match_count_experiment_plan.csv"
)

opponent_allocation_file = (
    REPORT_DIR
    / "opponent_allocation_plan.csv"
)

move_frequency_file = (
    REPORT_DIR
    / "move_frequency.csv"
)

state_frequency_file = (
    REPORT_DIR
    / "state_frequency.csv"
)

with open(
    analysis_summary_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        analysis_summary,
        file,
        indent=2,
    )

experiment_plan.to_csv(
    experiment_plan_file,
    index=False,
)

opponent_allocation.to_csv(
    opponent_allocation_file,
    index=False,
)

move_counts.to_csv(
    move_frequency_file,
    index=False,
)

state_counts.rename("count").to_csv(
    state_frequency_file,
)

print("Exported:")
print(analysis_summary_file)
print(experiment_plan_file)
print(opponent_allocation_file)
print(move_frequency_file)
print(state_frequency_file)

Exported:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\replay_analysis_summary.json
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\match_count_experiment_plan.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\opponent_allocation_plan.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\move_frequency.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\state_frequency.csv


# Cell 16 — Validate Exports

In [15]:
expected_files = [
    analysis_summary_file,
    experiment_plan_file,
    opponent_allocation_file,
    move_frequency_file,
    state_frequency_file,
]

missing_files = [
    path
    for path in expected_files
    if not path.exists()
]

assert not missing_files, missing_files

for path in expected_files:
    print(
        f"[OK] {path.name:<36} "
        f"{path.stat().st_size:>8,} bytes"
    )

print()
print(
    "All Notebook 29 reports exported successfully."
)

[OK] replay_analysis_summary.json              531 bytes
[OK] match_count_experiment_plan.csv           202 bytes
[OK] opponent_allocation_plan.csv              275 bytes
[OK] move_frequency.csv                         96 bytes
[OK] state_frequency.csv                     1,030 bytes

All Notebook 29 reports exported successfully.


# Cell 17 — Configurable Benchmark Configuration

In [16]:
BENCHMARK_MATCH_COUNTS = [
    100,
    1_000,
    2_000,
]

print("=" * 70)
print("SELF-PLAY SCALE BENCHMARK")
print("=" * 70)

for match_count in BENCHMARK_MATCH_COUNTS:
    print(f"Planned benchmark: {match_count:,} matches")

assert BENCHMARK_MATCH_COUNTS == [100, 1_000, 2_000]

SELF-PLAY SCALE BENCHMARK
Planned benchmark: 100 matches
Planned benchmark: 1,000 matches
Planned benchmark: 2,000 matches


## Cell 18 — Inspect the Existing Self-Play API

#### Before running thousands of games, let’s confirm the exact reusable functions and configuration classes exported by Notebook 28.

In [17]:
import inspect
import sys

from pathlib import Path


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            (candidate / "src").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located."
    )


PROJECT_ROOT = find_project_root()

project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(
        0,
        project_root_str,
    )

print("Project root:", PROJECT_ROOT)
print("Python path updated:", project_root_str in sys.path)

import src.training.self_play as self_play_module


print("=" * 70)
print("SELF-PLAY MODULE INSPECTION")
print("=" * 70)

public_names = [
    name
    for name in dir(self_play_module)
    if not name.startswith("_")
]

print("Public names:")
for name in public_names:
    print(f" - {name}")

print()

if hasattr(
    self_play_module,
    "run_self_play_session",
):
    run_self_play_session = (
        self_play_module.run_self_play_session
    )

    print("run_self_play_session signature:")
    print(
        inspect.signature(
            run_self_play_session
        )
    )

    print()
    print("Function located successfully.")
else:
    raise ImportError(
        "run_self_play_session was not found "
        "in src.training.self_play."
    )

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Python path updated: True
SELF-PLAY MODULE INSPECTION
Public names:
 - ReplayGame
 - ReplayRecorder
 - SelfPlayRunSummary
 - TournamentMatch
 - annotations
 - dataclass
 - run_self_play_session
 - run_two_agent_match
 - uuid

run_self_play_session signature:
(*, match_factory, initial_state_factory, num_games: 'int', verbose: 'bool' = False) -> 'SelfPlayRunSummary'

Function located successfully.


# Cell 19 — Inspect Configuration Classes

In [18]:
possible_config_names = [
    "SelfPlayConfig",
    "TrainingConfig",
    "SelfPlaySessionConfig",
]

available_config_classes = {}

for name in possible_config_names:
    value = getattr(
        self_play_module,
        name,
        None,
    )

    if value is not None:
        available_config_classes[name] = value

print("=" * 70)
print("AVAILABLE CONFIGURATION CLASSES")
print("=" * 70)

if available_config_classes:
    for name, config_class in available_config_classes.items():
        print(f"{name}:")
        print(inspect.signature(config_class))
        print()
else:
    print(
        "No expected configuration class name "
        "was found."
    )

    print()
    print(
        "The benchmark runner will be adapted "
        "to the actual function signature above."
    )

AVAILABLE CONFIGURATION CLASSES
No expected configuration class name was found.

The benchmark runner will be adapted to the actual function signature above.


# Cell 20 — Benchmark Runner

In [19]:
import inspect

import src.training.self_play as self_play_module


print("=" * 70)
print("SELF-PLAY FACTORY DISCOVERY")
print("=" * 70)

candidate_modules = [
    self_play_module,
]

candidate_names = [
    "match_factory",
    "initial_state_factory",
    "build_match_factory",
    "build_initial_state",
    "create_match",
    "create_initial_state",
]

for module in candidate_modules:
    print(f"\nModule: {module.__name__}")

    for name in candidate_names:
        value = getattr(module, name, None)

        if value is not None:
            print(f" - {name}: {value}")

            if callable(value):
                try:
                    print(
                        "   signature:",
                        inspect.signature(value),
                    )
                except (TypeError, ValueError):
                    pass

SELF-PLAY FACTORY DISCOVERY

Module: src.training.self_play


# Cell 21

In [20]:
from pathlib import Path

SELF_PLAY_FILE = (
    PROJECT_ROOT
    / "src"
    / "training"
    / "self_play.py"
)

print("Self-play file:", SELF_PLAY_FILE)
print("Exists:", SELF_PLAY_FILE.exists())

print()
print(SELF_PLAY_FILE.read_text(encoding="utf-8"))

Self-play file: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\training\self_play.py
Exists: True

﻿from __future__ import annotations

import uuid
from dataclasses import dataclass

from src.tournament.match import TournamentMatch
from src.tournament.runner import run_two_agent_match

from .replay import ReplayGame, ReplayRecorder


@dataclass(slots=True)
class SelfPlayRunSummary:
    requested_games: int
    completed_games: int
    recorder: ReplayRecorder


def run_self_play_session(
    *,
    match_factory,
    initial_state_factory,
    num_games: int,
    verbose: bool = False,
) -> SelfPlayRunSummary:
    """
    Run repeated self-play matches using caller-provided factories.

    Parameters
    ----------
    match_factory:
        Callable accepting game_index and returning TournamentMatch.

    initial_state_factory:
        Callable accepting game_index and returning BattleState.

    num_games:
        Number of matches to execute.
    """

    if num_gam

## Cell - 22

In [21]:
import pandas as pd


benchmark_plan = pd.DataFrame(
    [
        {
            "phase": "Baseline validation",
            "games": 100,
            "status": "Completed in Notebook 28",
        },
        {
            "phase": "Scaling benchmark",
            "games": 1_000,
            "status": "Planned after opponent diversity",
        },
        {
            "phase": "Full training benchmark",
            "games": 2_000,
            "status": "Planned after opponent diversity",
        },
    ]
)

display(benchmark_plan)

,phase,games,status
0,Baseline validation,100,Completed in Notebook 28
1,Scaling benchmark,1000,Planned after opponent diversity
2,Full training benchmark,2000,Planned after opponent diversity


## Cell 23

In [22]:
benchmark_plan_file = (
    REPORT_DIR
    / "benchmark_execution_plan.csv"
)

benchmark_plan.to_csv(
    benchmark_plan_file,
    index=False,
)

print("[OK]", benchmark_plan_file)

[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook29\benchmark_execution_plan.csv


## Cell 24

In [23]:
expected_report_names = [
    "replay_analysis_summary.json",
    "match_count_experiment_plan.csv",
    "opponent_allocation_plan.csv",
    "move_frequency.csv",
    "state_frequency.csv",
    "benchmark_execution_plan.csv",
]

missing_reports = [
    name
    for name in expected_report_names
    if not (REPORT_DIR / name).exists()
]

assert not missing_reports, missing_reports

for name in expected_report_names:
    path = REPORT_DIR / name

    print(
        f"[OK] {name:<38} "
        f"{path.stat().st_size:>8,} bytes"
    )

print()
print("All six Notebook 29 reports validated.")

[OK] replay_analysis_summary.json                531 bytes
[OK] match_count_experiment_plan.csv             202 bytes
[OK] opponent_allocation_plan.csv                275 bytes
[OK] move_frequency.csv                           96 bytes
[OK] state_frequency.csv                       1,030 bytes
[OK] benchmark_execution_plan.csv                190 bytes

All six Notebook 29 reports validated.
